# BÀI THỰC HÀNH 4: MẠNG NEURAL HỒI QUY

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import json
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

DATA_DIR = Path("/content/drive/MyDrive/PhoMT")
TRAIN_FILE = DATA_DIR / "small-train.json"
DEV_FILE = DATA_DIR / "small-dev.json"
TEST_FILE = DATA_DIR / "small-test.json"


Using device: cuda


In [4]:
# Tiền xử lý: xây dựng vocab đơn giản và tokenizer

PAD_TOKEN = "<pad>"
BOS_TOKEN = "<bos>"
EOS_TOKEN = "<eos>"
UNK_TOKEN = "<unk>"

class SimpleVocab:
    def __init__(self, texts, min_freq=2):
        self.special_tokens = [PAD_TOKEN, BOS_TOKEN, EOS_TOKEN, UNK_TOKEN]
        self.token2idx = {tok: i for i, tok in enumerate(self.special_tokens)}
        self.idx2token = list(self.special_tokens)

        freq = {}
        for sent in texts:
            for tok in sent.split():
                freq[tok] = freq.get(tok, 0) + 1
        for tok, c in freq.items():
            if c >= min_freq and tok not in self.token2idx:
                self.token2idx[tok] = len(self.idx2token)
                self.idx2token.append(tok)

        self.pad_idx = self.token2idx[PAD_TOKEN]
        self.bos_idx = self.token2idx[BOS_TOKEN]
        self.eos_idx = self.token2idx[EOS_TOKEN]
        self.unk_idx = self.token2idx[UNK_TOKEN]

    def encode(self, text, add_bos_eos=True):
        ids = []
        if add_bos_eos:
            ids.append(self.bos_idx)
        for tok in text.split():
            ids.append(self.token2idx.get(tok, self.unk_idx))
        if add_bos_eos:
            ids.append(self.eos_idx)
        return ids

    def decode(self, ids):
        toks = []
        for i in ids:
            if i == self.eos_idx:
                break
            if i in (self.bos_idx, self.pad_idx):
                continue
            toks.append(self.idx2token[i])
        return " ".join(toks)

    @property
    def size(self):
        return len(self.idx2token)


def load_phomt_json(path: Path):
    data = []
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    src_texts = [ex["english"].strip() for ex in data]
    tgt_texts = [ex["vietnamese"].strip() for ex in data]
    return src_texts, tgt_texts


train_src, train_tgt = load_phomt_json(TRAIN_FILE)
dev_src, dev_tgt = load_phomt_json(DEV_FILE)

src_vocab = SimpleVocab(train_src)
tgt_vocab = SimpleVocab(train_tgt)

print("#src vocab =", src_vocab.size)
print("#tgt vocab =", tgt_vocab.size)


#src vocab = 10721
#tgt vocab = 5080


In [5]:
class PhoMTDataset(Dataset):
    def __init__(self, src_texts, tgt_texts, src_vocab: SimpleVocab, tgt_vocab: SimpleVocab, max_len=80):
        self.src_ids = [src_vocab.encode(s, add_bos_eos=False)[:max_len] for s in src_texts]
        self.tgt_ids = [tgt_vocab.encode(t, add_bos_eos=True)[:max_len] for t in tgt_texts]
        assert len(self.src_ids) == len(self.tgt_ids)
        self.pad_idx_src = src_vocab.pad_idx
        self.pad_idx_tgt = tgt_vocab.pad_idx

    def __len__(self):
        return len(self.src_ids)

    def __getitem__(self, idx):
        return torch.tensor(self.src_ids[idx], dtype=torch.long), torch.tensor(self.tgt_ids[idx], dtype=torch.long)


def collate_fn(batch):
    src_seqs, tgt_seqs = zip(*batch)
    src_lens = [len(s) for s in src_seqs]
    tgt_lens = [len(t) for t in tgt_seqs]

    max_src = max(src_lens)
    max_tgt = max(tgt_lens)

    pad_src = src_vocab.pad_idx
    pad_tgt = tgt_vocab.pad_idx

    src_batch = torch.full((len(batch), max_src), pad_src, dtype=torch.long)
    tgt_batch = torch.full((len(batch), max_tgt), pad_tgt, dtype=torch.long)

    for i, (s, t) in enumerate(zip(src_seqs, tgt_seqs)):
        src_batch[i, : len(s)] = s
        tgt_batch[i, : len(t)] = t

    return src_batch, tgt_batch


train_dataset = PhoMTDataset(train_src, train_tgt, src_vocab, tgt_vocab)
dev_dataset = PhoMTDataset(dev_src, dev_tgt, src_vocab, tgt_vocab)

BATCH_SIZE = 32
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
dev_loader = DataLoader(dev_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

len(train_dataset), len(dev_dataset)


(20000, 2000)

## Bài 1:
Xây dựng kiến trúc Encoder-Decoder gồm 3 lớp LSTM cho module encoder và 3 lớp LSTM cho module decoder, với hidden size là 256, cho bài toán dịch máy từ tiếng Anh sang tiếng Việt. Huấn luyện mô hình này trên bộ dữ liệu PhoMT sử dụng Adam làm phương thức tối ưu tham số. Đánh giá độ hiệu quả của mô hình sử dụng độ đo ROUGE-L.

In [6]:
EMB_DIM = 256
HIDDEN_SIZE = 256
NUM_LAYERS = 3

class Encoder3LSTM(nn.Module):
    def __init__(self, vocab_size, emb_dim=EMB_DIM, hidden_size=HIDDEN_SIZE, num_layers=NUM_LAYERS):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=src_vocab.pad_idx)
        self.lstm = nn.LSTM(
            input_size=emb_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
        )

    def forward(self, src):
        embedded = self.embedding(src)
        outputs, (hidden, cell) = self.lstm(embedded)
        return outputs, (hidden, cell)


class Decoder3LSTM(nn.Module):
    def __init__(self, vocab_size, emb_dim=EMB_DIM, hidden_size=HIDDEN_SIZE, num_layers=NUM_LAYERS):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=tgt_vocab.pad_idx)
        self.lstm = nn.LSTM(
            input_size=emb_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
        )
        self.fc_out = nn.Linear(hidden_size, vocab_size)

    def forward(self, input_token, hidden, cell):
        # input_token: [batch, 1]
        embedded = self.embedding(input_token)
        output, (hidden, cell) = self.lstm(embedded, (hidden, cell))
        prediction = self.fc_out(output.squeeze(1))
        return prediction, hidden, cell

In [7]:
class Seq2SeqNoAttn(nn.Module):
    def __init__(self, encoder, decoder, pad_idx):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.pad_idx = pad_idx

    def forward(self, src, tgt, teacher_forcing_ratio=0.5):
        batch_size, tgt_len = tgt.shape
        vocab_size = self.decoder.fc_out.out_features

        _, (hidden, cell) = self.encoder(src)

        outputs = torch.zeros(batch_size, tgt_len - 1, vocab_size, device=src.device)

        input_token = tgt[:, 0].unsqueeze(1)  # <bos>
        for t in range(1, tgt_len):
            prediction, hidden, cell = self.decoder(input_token, hidden, cell)
            outputs[:, t - 1, :] = prediction

            teacher_force = np.random.random() < teacher_forcing_ratio
            top1 = prediction.argmax(dim=1)
            input_token = (tgt[:, t] if teacher_force else top1).unsqueeze(1)

        return outputs

    def translate(self, src, max_len=60):
        self.eval()
        with torch.no_grad():
            batch_size = src.size(0)
            _, (hidden, cell) = self.encoder(src)
            input_token = torch.full((batch_size, 1), tgt_vocab.bos_idx, dtype=torch.long, device=src.device)
            outputs = []
            for _ in range(max_len):
                prediction, hidden, cell = self.decoder(input_token, hidden, cell)
                top1 = prediction.argmax(dim=1)
                outputs.append(top1.unsqueeze(1))
                input_token = top1.unsqueeze(1)
            outputs = torch.cat(outputs, dim=1)
        return outputs

In [12]:
encoder1 = Encoder3LSTM(src_vocab.size).to(DEVICE)
decoder1 = Decoder3LSTM(tgt_vocab.size).to(DEVICE)
model1 = Seq2SeqNoAttn(encoder1, decoder1, tgt_vocab.pad_idx).to(DEVICE)

criterion = nn.CrossEntropyLoss(ignore_index=tgt_vocab.pad_idx)
optimizer = torch.optim.Adam(model1.parameters(), lr=1e-3)


In [13]:
# Hàm tính ROUGE-L đơn giản (theo LCS)

def lcs_length(x_tokens, y_tokens):
    m, n = len(x_tokens), len(y_tokens)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if x_tokens[i - 1] == y_tokens[j - 1]:
                dp[i][j] = dp[i - 1][j - 1] + 1
            else:
                dp[i][j] = max(dp[i - 1][j], dp[i][j - 1])
    return dp[m][n]


def rouge_l_score(pred, ref):
    pred_tokens = pred.split()
    ref_tokens = ref.split()
    lcs = lcs_length(pred_tokens, ref_tokens)
    if lcs == 0:
        return 0.0
    prec = lcs / len(pred_tokens)
    rec = lcs / len(ref_tokens)
    if prec + rec == 0:
        return 0.0
    beta2 = 1.0
    score = (1 + beta2) * prec * rec / (rec + beta2 * prec)
    return score


def evaluate_rouge_l(model, data_loader, max_len=60, n_batches=5):
    model.eval()
    scores = []
    with torch.no_grad():
        for b_idx, (src, tgt) in enumerate(data_loader):
            src = src.to(DEVICE)
            tgt = tgt.to(DEVICE)
            pred_ids = model.translate(src, max_len=max_len)
            for i in range(src.size(0)):
                pred = tgt_vocab.decode(pred_ids[i].cpu().tolist())
                ref = tgt_vocab.decode(tgt[i].cpu().tolist())
                scores.append(rouge_l_score(pred, ref))
            if b_idx + 1 >= n_batches:
                break
    return float(np.mean(scores)) if scores else 0.0


In [14]:
# Train Bài 1

EPOCHS = 5

for epoch in range(1, EPOCHS + 1):
    model1.train()
    total_loss = 0.0
    for src, tgt in train_loader:
        src = src.to(DEVICE)
        tgt = tgt.to(DEVICE)

        optimizer.zero_grad()
        outputs = model1(src, tgt, teacher_forcing_ratio=0.5)  # [batch, tgt_len-1, vocab]
        # Dịch sang dạng [batch*(tgt_len-1), vocab]
        logits = outputs.reshape(-1, outputs.size(-1))
        # Target bỏ token đầu (bos)
        tgt_y = tgt[:, 1:].reshape(-1)
        loss = criterion(logits, tgt_y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model1.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    dev_rouge = evaluate_rouge_l(model1, dev_loader, max_len=60, n_batches=5)
    print(f"Epoch {epoch}: train loss = {avg_loss:.4f}, dev ROUGE-L = {dev_rouge:.4f}")


Epoch 1: train loss = 6.1760, dev ROUGE-L = 0.0309
Epoch 2: train loss = 5.8390, dev ROUGE-L = 0.0740
Epoch 3: train loss = 5.5691, dev ROUGE-L = 0.1017
Epoch 4: train loss = 5.3905, dev ROUGE-L = 0.1233
Epoch 5: train loss = 5.2530, dev ROUGE-L = 0.1293


## Bài 2:
Xây dựng kiến trúc Encoder-Decoder gồm 3 lớp LSTM cho module encoder và 3 lớp LSTM cho module decoder, với hidden size là 256, cho bài toán dịch máy từ tiếng Anh sang tiếng Việt. Module decoder được trang bị kỹ thuật attention theo mô tả của nghiên cứu "[Neural Machine Translation by Jointly Learning to Align and Translate](https://arxiv.org/abs/1409.0473)". Huấn luyện mô hình này trên bộ dữ liệu PhoMT sử dụng Adam làm phương thức tối ưu tham số. Đánh giá độ hiệu quả của mô hình sử dụn độ đo ROUGE-L.

In [15]:
class BahdanauAttention(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.W_q = nn.Linear(hidden_size, hidden_size, bias=False)
        self.W_k = nn.Linear(hidden_size, hidden_size, bias=False)
        self.v = nn.Linear(hidden_size, 1, bias=False)

    def forward(self, decoder_hidden, encoder_outputs):
        """decoder_hidden: [batch, hidden]; encoder_outputs: [batch, src_len, hidden]"""
        query = self.W_q(decoder_hidden).unsqueeze(1)          # [B, 1, H]
        keys = self.W_k(encoder_outputs)                       # [B, S, H]
        energy = torch.tanh(query + keys)                      # [B, S, H]
        scores = self.v(energy).squeeze(-1)                    # [B, S]
        attn_weights = torch.softmax(scores, dim=-1)           # [B, S]
        context = torch.bmm(attn_weights.unsqueeze(1), encoder_outputs).squeeze(1)  # [B, H]
        return context, attn_weights


class DecoderBahdanau(nn.Module):
    def __init__(self, vocab_size, emb_dim=EMB_DIM, hidden_size=HIDDEN_SIZE, num_layers=NUM_LAYERS, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=tgt_vocab.pad_idx)
        self.lstm = nn.LSTM(
            input_size=emb_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.attention = BahdanauAttention(hidden_size)
        self.fc = nn.Linear(hidden_size * 2, vocab_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, input_token, hidden, cell, encoder_outputs):
        embedded = self.dropout(self.embedding(input_token))   # [B, 1, E]
        output, (hidden, cell) = self.lstm(embedded, (hidden, cell))  # output: [B, 1, H]
        dec_last = hidden[-1]                                  # [B, H]
        context, attn_weights = self.attention(dec_last, encoder_outputs)
        combined = torch.cat([output.squeeze(1), context], dim=-1)    # [B, 2H]
        prediction = self.fc(combined)                         # [B, vocab]
        return prediction, hidden, cell, attn_weights

In [16]:
class Seq2SeqBahdanau(nn.Module):
    def __init__(self, encoder, decoder, pad_idx):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.pad_idx = pad_idx

    def forward(self, src, tgt, teacher_forcing_ratio=0.5):
        batch_size, tgt_len = tgt.shape
        vocab_size = self.decoder.fc.out_features

        enc_outputs, (hidden, cell) = self.encoder(src)  # enc_outputs: [B, S, H]
        outputs = torch.zeros(batch_size, tgt_len - 1, vocab_size, device=src.device)

        input_token = tgt[:, 0].unsqueeze(1)
        for t in range(1, tgt_len):
            prediction, hidden, cell, _ = self.decoder(input_token, hidden, cell, enc_outputs)
            outputs[:, t - 1, :] = prediction

            teacher_force = np.random.random() < teacher_forcing_ratio
            top1 = prediction.argmax(dim=1)
            input_token = (tgt[:, t] if teacher_force else top1).unsqueeze(1)
        return outputs

    def translate(self, src, max_len=60):
        self.eval()
        with torch.no_grad():
            batch_size = src.size(0)
            enc_outputs, (hidden, cell) = self.encoder(src)
            input_token = torch.full((batch_size, 1), tgt_vocab.bos_idx, dtype=torch.long, device=src.device)
            outputs = []
            for _ in range(max_len):
                prediction, hidden, cell, _ = self.decoder(input_token, hidden, cell, enc_outputs)
                top1 = prediction.argmax(dim=1)
                outputs.append(top1.unsqueeze(1))
                input_token = top1.unsqueeze(1)
            outputs = torch.cat(outputs, dim=1)
        return outputs

In [17]:
# Khởi tạo model cho Bài 2 (dùng lại Encoder3LSTM)
encoder2 = Encoder3LSTM(src_vocab.size).to(DEVICE)
decoder2 = DecoderBahdanau(tgt_vocab.size).to(DEVICE)
model2 = Seq2SeqBahdanau(encoder2, decoder2, tgt_vocab.pad_idx).to(DEVICE)

criterion2 = nn.CrossEntropyLoss(ignore_index=tgt_vocab.pad_idx)
optimizer2 = torch.optim.Adam(model2.parameters(), lr=1e-3)


In [22]:
EPOCHS2 = 5

for epoch in range(1, EPOCHS2 + 1):
    model2.train()
    total_loss = 0.0
    for src, tgt in train_loader:
        src = src.to(DEVICE)
        tgt = tgt.to(DEVICE)

        optimizer2.zero_grad()
        outputs = model2(src, tgt, teacher_forcing_ratio=0.5)
        logits = outputs.reshape(-1, outputs.size(-1))
        tgt_y = tgt[:, 1:].reshape(-1)
        loss = criterion2(logits, tgt_y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model2.parameters(), 1.0)
        optimizer2.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    dev_rouge = evaluate_rouge_l(model2, dev_loader, max_len=60, n_batches=5)
    print(f"[Bahdanau] Epoch {epoch}: train loss = {avg_loss:.4f}, dev ROUGE-L = {dev_rouge:.4f}")


[Bahdanau] Epoch 1: train loss = 6.1249, dev ROUGE-L = 0.0601
[Bahdanau] Epoch 2: train loss = 5.6308, dev ROUGE-L = 0.1635
[Bahdanau] Epoch 3: train loss = 5.2005, dev ROUGE-L = 0.1990
[Bahdanau] Epoch 4: train loss = 4.8460, dev ROUGE-L = 0.2288
[Bahdanau] Epoch 5: train loss = 4.5400, dev ROUGE-L = 0.2306


## Bài 3:
Xây dựng kiến trúc Encoder-Decoder gồm 3 lớp LSTM cho module encoder và 3 lớp LSTM cho module decoder, với hidden size là 256, cho bài toán dịch máy từ tiếng Anh sang tiếng Việt. Module decoder được trang bị kỹ thuật attention theo mô tả của nghiên cứu "[Effective Approaches to Attention-based Neural Machine Translation](https://arxiv.org/abs/1508.04025)". Huấn luyện mô hình này trên bộ dữ liệu PhoMT sử dụng Adam làm phương thức tối ưu tham số. Đánh giá độ hiệu quả của mô hình sử dụn độ đo ROUGE-L.

In [18]:
# Bài 3: Encoder-Decoder 3 LSTM với Luong Attention (theo paper 2015)

class LuongAttention(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.W = nn.Linear(hidden_size, hidden_size, bias=False)

    def forward(self, decoder_hidden, encoder_outputs):
        """decoder_hidden: [B, H]; encoder_outputs: [B, S, H]"""
        query = decoder_hidden.unsqueeze(1)              # [B, 1, H]
        keys = self.W(encoder_outputs)                   # [B, S, H]
        scores = torch.bmm(query, keys.transpose(1, 2)).squeeze(1)  # [B, S]
        attn_weights = torch.softmax(scores, dim=-1)     # [B, S]
        context = torch.bmm(attn_weights.unsqueeze(1), encoder_outputs).squeeze(1)  # [B, H]
        return context, attn_weights


class DecoderLuong(nn.Module):
    def __init__(self, vocab_size, emb_dim=EMB_DIM, hidden_size=HIDDEN_SIZE, num_layers=NUM_LAYERS, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=tgt_vocab.pad_idx)
        self.lstm = nn.LSTM(
            input_size=emb_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.attention = LuongAttention(hidden_size)
        self.fc = nn.Linear(hidden_size * 2, vocab_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, input_token, hidden, cell, encoder_outputs):
        embedded = self.dropout(self.embedding(input_token))       # [B, 1, E]
        output, (hidden, cell) = self.lstm(embedded, (hidden, cell))   # output: [B, 1, H]
        dec_last = hidden[-1]                                      # [B, H]
        context, attn_weights = self.attention(dec_last, encoder_outputs)
        combined = torch.cat([output.squeeze(1), context], dim=-1) # [B, 2H]
        prediction = self.fc(combined)                             # [B, vocab]
        return prediction, hidden, cell, attn_weights


In [19]:
class Seq2SeqLuong(nn.Module):
    def __init__(self, encoder, decoder, pad_idx):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.pad_idx = pad_idx

    def forward(self, src, tgt, teacher_forcing_ratio=0.5):
        batch_size, tgt_len = tgt.shape
        vocab_size = self.decoder.fc.out_features

        enc_outputs, (hidden, cell) = self.encoder(src)
        outputs = torch.zeros(batch_size, tgt_len - 1, vocab_size, device=src.device)

        input_token = tgt[:, 0].unsqueeze(1)
        for t in range(1, tgt_len):
            prediction, hidden, cell, _ = self.decoder(input_token, hidden, cell, enc_outputs)
            outputs[:, t - 1, :] = prediction

            teacher_force = np.random.random() < teacher_forcing_ratio
            top1 = prediction.argmax(dim=1)
            input_token = (tgt[:, t] if teacher_force else top1).unsqueeze(1)
        return outputs

    def translate(self, src, max_len=60):
        self.eval()
        with torch.no_grad():
            batch_size = src.size(0)
            enc_outputs, (hidden, cell) = self.encoder(src)
            input_token = torch.full((batch_size, 1), tgt_vocab.bos_idx, dtype=torch.long, device=src.device)
            outputs = []
            for _ in range(max_len):
                prediction, hidden, cell, _ = self.decoder(input_token, hidden, cell, enc_outputs)
                top1 = prediction.argmax(dim=1)
                outputs.append(top1.unsqueeze(1))
                input_token = top1.unsqueeze(1)
            outputs = torch.cat(outputs, dim=1)
        return outputs

In [20]:
# Khởi tạo model cho Bài 3
encoder3 = Encoder3LSTM(src_vocab.size).to(DEVICE)
decoder3 = DecoderLuong(tgt_vocab.size).to(DEVICE)
model3 = Seq2SeqLuong(encoder3, decoder3, tgt_vocab.pad_idx).to(DEVICE)

criterion3 = nn.CrossEntropyLoss(ignore_index=tgt_vocab.pad_idx)
optimizer3 = torch.optim.Adam(model3.parameters(), lr=1e-3)

In [21]:
EPOCHS3 = 5

for epoch in range(1, EPOCHS3 + 1):
    model3.train()
    total_loss = 0.0
    for src, tgt in train_loader:
        src = src.to(DEVICE)
        tgt = tgt.to(DEVICE)

        optimizer3.zero_grad()
        outputs = model3(src, tgt, teacher_forcing_ratio=0.5)
        logits = outputs.reshape(-1, outputs.size(-1))
        tgt_y = tgt[:, 1:].reshape(-1)
        loss = criterion3(logits, tgt_y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model3.parameters(), 1.0)
        optimizer3.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    dev_rouge = evaluate_rouge_l(model3, dev_loader, max_len=60, n_batches=5)
    print(f"[Luong] Epoch {epoch}: train loss = {avg_loss:.4f}, dev ROUGE-L = {dev_rouge:.4f}")


[Luong] Epoch 1: train loss = 6.0945, dev ROUGE-L = 0.0788
[Luong] Epoch 2: train loss = 5.6514, dev ROUGE-L = 0.1383
[Luong] Epoch 3: train loss = 5.3246, dev ROUGE-L = 0.1730
[Luong] Epoch 4: train loss = 5.0277, dev ROUGE-L = 0.1865
[Luong] Epoch 5: train loss = 4.7781, dev ROUGE-L = 0.2129
